# BME i9400 — Meeting 8
## Linear Classification I: Logistic Regression

**Fall 2026 · Wednesday, October 7**

Today's lecture is about classification, which is the prediction of which of two categories a
sample belongs to. It develops logistic regression by the route of Meeting 6 — a model, a
likelihood, a loss, and a gradient — and introduces the decision boundary that a linear classifier
draws through the space of features.

**Agenda**

1. The classification problem
2. Why not least squares
3. From a linear score to a probability
4. The likelihood and the cross-entropy loss
5. Decision boundaries
6. Fitting and reading the full model
7. Connection to the course

## 1 — The classification problem

In classification the target takes one of a fixed set of values. Today the set has two elements:

$$y_i \in \{0, 1\}$$

> The class labelled $1$ is called the **positive class** — by convention the condition of interest:
> the disease present, the event occurred. The class labelled $0$ is the **negative class**. Which
> outcome is called positive is a choice, and every quantity in Meeting 9 depends on it.

Two kinds of output are useful. A **decision** assigns each new sample to a class. A
**probability** $P(y = 1 \mid x)$ states how likely the positive class is for a sample with features
$x$; it is the posterior probability of Meeting 2, with the features in place of the test result. A
probability can always be turned into a decision by choosing a threshold; a decision cannot be turned
back into a probability. Logistic regression produces the probability.

### The running example

The dataset is from a study of diabetes among women of Pima heritage near Phoenix, Arizona, conducted
by the National Institute of Diabetes and Digestive and Kidney Diseases (Smith, Everhart, Dickson,
Knowler and Johannes, *Proceedings of the Symposium on Computer Applications in Medical Care*, 1988).
It is in the course repository as `data/diabetes.csv`.

- $n = 768$ women, aged 21 and over
- $y$: whether diabetes was diagnosed within five years of the examination, by WHO criteria
- eight measurements taken at the examination:
    - number of pregnancies
    - plasma glucose concentration two hours into an oral glucose tolerance test (mg/dL)
    - diastolic blood pressure (mm Hg)
    - triceps skin-fold thickness (mm)
    - two-hour serum insulin (μU/mL)
    - body mass index (kg/m²)
    - a diabetes pedigree function, which scores family history
    - age (years)

### A data readiness decision

In this file a value of zero was recorded where a measurement is missing. A glucose concentration,
a blood pressure, or a body mass index of zero is not possible, so these zeros can be identified.

- glucose, blood pressure, and BMI are missing for a few percent of the women; those rows are removed
- skin-fold thickness and insulin are missing for 30% and 49% of the women; these two columns are
  set aside, and the lecture uses the remaining six measurements

After these steps $n = 724$ women remain, of whom 249 (34.4%) developed diabetes. Meeting 10 returns
to the two columns that were set aside.

> The **prevalence** in this dataset is the fraction of women with the positive label, 34.4%. It is
> far higher than the prevalence of diabetes in the general population, because the study population
> was selected for high risk. Every quantity that depends on prevalence (Meeting 2) is therefore
> specific to this population.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12, "figure.dpi": 55})
%config InlineBackend.figure_format = "retina"       # slides: figures display at half size, double resolution

URL = "https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/diabetes.csv"
raw = pd.read_csv(URL)

print(f"rows in the file: {len(raw)}")
print("zeros recorded in each measurement:")
for col in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]:
    print(f"  {col:<14} {(raw[col] == 0).sum():4d}   ({100*(raw[col] == 0).mean():.1f}%)")

In [ ]:
df = raw[(raw["Glucose"] > 0) & (raw["BloodPressure"] > 0) & (raw["BMI"] > 0)].reset_index(drop=True)
features = ["Pregnancies", "Glucose", "BloodPressure", "BMI", "DiabetesPedigreeFunction", "Age"]
X = df[features].values
y = df["Outcome"].values

print(f"\nafter removing rows with missing glucose, blood pressure, or BMI: n = {len(df)}")
print(f"positive class (diabetes within five years): {y.sum()} women, prevalence {100*y.mean():.1f}%")
df[features + ["Outcome"]].head(4)

In [ ]:
glucose = df["Glucose"].values
edges   = np.arange(50, 210, 10)                                 # 10 mg/dL bins
centers = edges[:-1] + 5
which   = np.digitize(glucose, edges) - 1
freq    = np.array([y[which == k].mean() if (which == k).sum() >= 5 else np.nan for k in range(len(centers))])

rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(glucose, y + rng.uniform(-0.04, 0.04, len(y)), s=8, alpha=0.3, color="#6e6878",
           label="one woman (label jittered)")
ax.plot(centers, freq, "o-", color="#c1272d", lw=2, ms=7, label="fraction with diabetes, per 10 mg/dL bin")
ax.set_xlabel("two-hour plasma glucose (mg/dL)"); ax.set_ylabel("outcome  /  fraction positive")
ax.set_yticks([0, 0.25, 0.5, 0.75, 1]); ax.set_title("Diabetes within five years against glucose")
ax.legend(loc="center left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Each grey point is one woman, drawn at her glucose concentration and her label. The red curve is the
fraction of women with diabetes in each 10 mg/dL bin: it rises from near zero below 90 mg/dL to
about 0.9 above 180 mg/dL, in the shape of an S. The task of the lecture is to model that curve —
$P(y = 1 \mid \text{glucose})$ as a function of glucose — and then to do the same with all six
measurements at once.

## 2 — Why not least squares

Meeting 6 fitted a line to a continuous target. The same line can be fitted to a 0/1 target and the
fitted value read as a probability. Three things go wrong.

- **The output is not a probability.** A line is unbounded: for this data it predicts a value below
  $0$ at a glucose of 60 mg/dL, and it would exceed $1$ beyond the range of the data.
- **The loss is wrong for the task.** A squared error treats a fitted value of $3$ for a woman with
  $y = 1$ as a large mistake, although it is a confident and correct classification.
- **The shape is wrong.** The fraction positive follows an S-shaped curve, flat near $0$ and near
  $1$; a line cannot be flat at both ends.

Two things are therefore needed: an output confined to the interval $[0, 1]$, and a loss derived
for binary targets. Sections 3 and 4 supply them.

In [ ]:
from sklearn.linear_model import LinearRegression

line = LinearRegression().fit(glucose.reshape(-1, 1), y)
for gv in [60, 100, 140, 199]:
    print(f"least squares fitted value at glucose = {gv:3d} mg/dL: {line.predict([[gv]])[0]:+.3f}")

In [ ]:
grid = np.linspace(40, 220, 200)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(centers, freq, "o", color="#c1272d", ms=7, label="fraction with diabetes, per bin")
ax.plot(grid, line.predict(grid.reshape(-1, 1)), color="#2b6cb0", lw=2.5, label="least squares line")
ax.axhline(0, color="k", lw=0.8); ax.axhline(1, color="k", lw=0.8)
ax.set_xlabel("two-hour plasma glucose (mg/dL)"); ax.set_ylabel("fraction positive  /  fitted value")
ax.set_title("A straight line fitted to the 0/1 labels")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3 — From a linear score to a probability

Keep the linear part of Meeting 6. With the design-matrix convention — a column of ones in front
of the features — each sample has a **linear score**

$$z_i = x_i^{\top} w = w_0 + w_1 x_{i1} + \cdots + w_d x_{id}$$

which can take any real value. The score is then passed through the **sigmoid function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

> The sigmoid, also called the *logistic function*, maps any real number to a number strictly between
> $0$ and $1$. Three properties are used below:
>
> - $\sigma(0) = 1/2$; a large positive $z$ gives a value near $1$, a large negative $z$ a value near $0$
> - $\sigma(-z) = 1 - \sigma(z)$, so the curve is symmetric about $z = 0$
> - its derivative is $\sigma'(z) = \sigma(z)\left(1 - \sigma(z)\right)$

### The logistic regression model

$$P(y = 1 \mid x) = \sigma\!\left(x^{\top} w\right), \qquad\qquad P(y = 0 \mid x) = 1 - \sigma\!\left(x^{\top} w\right) = \sigma\!\left(-x^{\top} w\right)$$

The model has the same parameters as linear regression — one weight per feature and an intercept —
and the same linear score. Only the final step differs: the score is converted into a probability
instead of being used directly as the prediction.

### Odds and log-odds

Solving the model for the score shows what the weights mean. Writing $p = P(y = 1 \mid x)$,

$$\frac{p}{1 - p} = e^{x^{\top} w}, \qquad\qquad \log \frac{p}{1 - p} = x^{\top} w$$

> $p / (1 - p)$ is the **odds** of the positive class, as in Meeting 2: a probability of $0.8$ is
> odds of $4$ to $1$. $\log\frac{p}{1-p}$ is the **log-odds**, also called the **logit**. The sigmoid
> is the inverse of the logit.

**Logistic regression is a linear model for the log-odds.** Each feature adds $w_j x_j$ to the
log-odds, and therefore multiplies the odds by $e^{w_j x_j}$. In Meeting 2 a positive test multiplied
the prior odds of disease by a likelihood ratio; here each unit of feature $j$ multiplies the odds by
$e^{w_j}$.

> $e^{w_j}$ is the **odds ratio** for feature $j$: the factor by which the odds of the positive class
> change when $x_j$ increases by one unit, with the other features held fixed. An odds ratio of $1$
> means that the feature does not change the odds; the corresponding weight is $0$.

In [ ]:
from sklearn.linear_model import LogisticRegression

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 400)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z, sigmoid(z), lw=2.5, color="#5b3a8e")
ax.axhline(0.5, ls="--", color="#6e6878", lw=1); ax.axvline(0, ls="--", color="#6e6878", lw=1)
ax.set_xlabel("linear score  $z = x^\\top w$"); ax.set_ylabel("$\\sigma(z)$"); ax.set_title("The sigmoid function")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Logistic regression with glucose as the only feature. penalty=None gives the maximum likelihood
# fit of section 4; by default scikit-learn adds a penalty on the weights (section 6).
model_1d = LogisticRegression(penalty=None, max_iter=1000).fit(glucose.reshape(-1, 1), y)
w0, w1   = model_1d.intercept_[0], model_1d.coef_[0, 0]

print(f"fitted model:  log-odds = {w0:.3f} + {w1:.4f} x glucose")
print(f"odds ratio per 1 mg/dL : {np.exp(w1):.4f}")
print(f"odds ratio per 10 mg/dL: {np.exp(10 * w1):.3f}")
print(f"P(diabetes) = 1/2 where the score is zero: glucose = {-w0 / w1:.1f} mg/dL\n")
for gv in [80, 100, 120, 140, 160, 180, 199]:
    print(f"  glucose {gv:3d} mg/dL:  P(diabetes within five years) = {sigmoid(w0 + w1 * gv):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(centers, freq, "o", color="#c1272d", ms=7, label="fraction with diabetes, per 10 mg/dL bin")
ax.plot(grid, sigmoid(w0 + w1 * grid), color="#5b3a8e", lw=2.5, label="logistic regression")
ax.plot(grid, line.predict(grid.reshape(-1, 1)), color="#2b6cb0", lw=1.5, ls="--", label="least squares line (section 2)")
ax.axvline(-w0 / w1, color="#6e6878", lw=1, ls=":")
ax.text(-w0 / w1 + 2, 0.05, f"$P = 1/2$ at {-w0/w1:.0f} mg/dL", color="#6e6878")
ax.set_ylim(-0.15, 1.1); ax.set_xlabel("two-hour plasma glucose (mg/dL)"); ax.set_ylabel("probability")
ax.set_title("The fitted probability of diabetes against glucose")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

The fitted curve follows the binned fractions across the whole range, including the flat ends that
the straight line could not reach. Two numbers from the fit:

- the odds ratio is $1.48$ per 10 mg/dL: each additional 10 mg/dL of two-hour glucose multiplies a
  woman's odds of developing diabetes by about one and a half
- the probability passes $1/2$ at 142 mg/dL, which is the WHO cut-off for impaired glucose
  tolerance (140 mg/dL two hours after the glucose load) — the model has recovered a clinical
  threshold from the labels alone

## 4 — The likelihood and the cross-entropy loss

Meeting 6 derived the squared loss from a Gaussian model of the noise. The same route — write
down a probability model for the labels, form the likelihood, take the logarithm, negate — gives the
loss for logistic regression.

The probability model is that each label is a coin flip whose probability of coming up $1$ is the
model's output:

$$y_i \sim \text{Bernoulli}(p_i), \qquad p_i = \sigma\!\left(x_i^{\top} w\right)$$

> A **Bernoulli** random variable takes the value $1$ with probability $p$ and $0$ with probability
> $1 - p$. The probability of an observed label $y_i$ can be written in one expression,
> $$P(y_i \mid x_i, w) = p_i^{\,y_i}\,(1 - p_i)^{\,1 - y_i}$$
> which equals $p_i$ when $y_i = 1$ and $1 - p_i$ when $y_i = 0$, because anything raised to the
> power $0$ is $1$.

### From likelihood to loss

With independent samples the likelihood is the product over the dataset,

$$p(y \mid X, w) = \prod_{i=1}^{n} p_i^{\,y_i}\,(1 - p_i)^{\,1 - y_i}$$

its logarithm is a sum,

$$\log p(y \mid X, w) = \sum_{i=1}^{n} \left[\, y_i \log p_i + (1 - y_i) \log (1 - p_i) \,\right]$$

and the loss is its negative:

$$\boxed{\; L(w) = -\sum_{i=1}^{n} \left[\, y_i \log p_i + (1 - y_i) \log (1 - p_i) \,\right], \qquad p_i = \sigma\!\left(x_i^{\top} w\right) \;}$$

### Reading the loss

> This is the **cross-entropy loss**, also called the *log loss* or the *negative log-likelihood*.
> The name comes from information theory, where the same expression measures the mismatch between
> two probability distributions.
>
> For a single sample only one of the two terms is nonzero, so the sample's loss is
> $-\log(\text{the probability the model assigned to the correct class})$. Assigning $0.9$ to the
> correct class costs $0.11$; assigning $0.5$ costs $0.69$; assigning $0.01$ costs $4.6$. Confident
> mistakes are expensive, and the cost grows without bound as the assigned probability approaches
> zero.

### The gradient

Differentiating the loss uses $\frac{d}{dz}\log\sigma(z) = 1 - \sigma(z)$ and
$\frac{d}{dz}\log\!\left(1 - \sigma(z)\right) = -\sigma(z)$, both consequences of
$\sigma' = \sigma(1 - \sigma)$. For sample $i$, with $z_i = x_i^{\top} w$ and
$\partial z_i / \partial w = x_i$,

$$\nabla_w \left[\, y_i \log p_i + (1 - y_i)\log(1 - p_i) \,\right] = \left[\, y_i (1 - p_i) - (1 - y_i)\, p_i \,\right] x_i = (y_i - p_i)\, x_i$$

Summing over the samples and negating,

$$\boxed{\; \nabla_w L = \sum_{i=1}^{n} (p_i - y_i)\, x_i = X^{\top}(p - y) \;}$$

> $p$ is the vector of the $n$ fitted probabilities. The gradient has the same form as in least
> squares, where it was $2X^{\top}(Xw - y)$: each sample's feature vector, weighted by the error of
> its prediction. Only the prediction has changed, from $x_i^{\top} w$ to $\sigma(x_i^{\top} w)$.

### No closed form

Setting the gradient to zero gives $X^{\top}\sigma(Xw) = X^{\top} y$. Because $\sigma$ is nonlinear,
these equations cannot be solved for $w$ algebraically, and there is no counterpart of the normal
equations. The minimum is found numerically, by starting from some $w$ and repeatedly stepping
against the gradient — **gradient descent**, the subject of Meeting 13. Two facts make this safe:

- the cross-entropy loss is **convex** in $w$, so it has a single minimum and no false ones
- the gradient costs one pass over the data to compute

> A **convex** function curves upward everywhere, like the bowl of Meeting 6: any local minimum is
> the global minimum, and gradient descent reaches it from any starting point.

Scikit-learn's `LogisticRegression` uses a refined form of this idea (the L-BFGS method) and
converges in a few dozen iterations. The cell below runs plain gradient descent on the glucose
model, to show that the same weights emerge.

In [ ]:
def cross_entropy(w, Xd, y):
    p = sigmoid(Xd @ w)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))      # per-sample average

def gradient(w, Xd, y):
    p = sigmoid(Xd @ w)
    return Xd.T @ (p - y) / len(y)                                  # X^T (p - y), averaged

# Standardise glucose first (Meeting 7): gradient descent is far easier on a standardised scale.
g_mean, g_sd = glucose.mean(), glucose.std(ddof=1)
Xd = np.column_stack([np.ones(len(glucose)), (glucose - g_mean) / g_sd])

In [ ]:
w, step, history = np.zeros(2), 0.5, []
for it in range(1, 301):
    w = w - step * gradient(w, Xd, y)                               # one step against the gradient
    history.append(cross_entropy(w, Xd, y))
    if it in (1, 5, 10, 25, 50, 100, 300):
        print(f"iteration {it:3d}:  loss {history[-1]:.4f}   w = ({w[0]:+.4f}, {w[1]:+.4f})")

check = LogisticRegression(penalty=None, max_iter=1000).fit(Xd[:, 1:], y)
print(f"\nscikit-learn:                      w = ({check.intercept_[0]:+.4f}, {check.coef_[0, 0]:+.4f})")
print(f"the same slope in mg/dL units: {w[1] / g_sd:.4f}   (section 3 found {w1:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 301), history, lw=2, color="#5b3a8e")
ax.set_xlabel("iteration (log scale)"); ax.set_ylabel("cross-entropy loss (per sample)")
ax.set_title("Gradient descent on the glucose model"); ax.set_xscale("log"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5 — Decision boundaries

A probability becomes a decision by comparing it with a threshold. With the threshold at $1/2$,

$$\text{predict } 1 \iff \sigma\!\left(x^{\top} w\right) \ge \tfrac{1}{2} \iff x^{\top} w \ge 0$$

because $\sigma(z) \ge 1/2$ exactly when $z \ge 0$. The set of points where the decision changes,
$x^{\top} w = 0$, is the **decision boundary**.

> With one feature the boundary is a point: 142 mg/dL in section 3. With two features it is a
> straight line; with $d$ features it is a flat surface of dimension $d - 1$, called a
> **hyperplane**. In every case it is flat, which is what makes logistic regression a *linear*
> classifier.

From Meeting 4, $x^{\top} w$ is the projection of $x$ onto the direction of $w$, scaled by $\|w\|$.
A linear classifier therefore does one thing: it projects every sample onto a single direction in
feature space and asks on which side of a cut the projection falls. The weight vector is
perpendicular to the boundary, and the lines of equal probability — $P = 0.25$, $P = 0.5$,
$P = 0.75$ — are parallel to it. How quickly the probability changes across the boundary is set by
$\|w\|$: a long weight vector gives a sharp transition, a short one a gradual one.

The threshold of $1/2$ is a convention, not a requirement. Moving it slides the boundary along $w$
without rotating it, and trades one kind of error for the other — the knob of Meeting 2. Meeting 9
is about choosing it.

In [ ]:
X2 = df[["Glucose", "BMI"]].values
model_2d = LogisticRegression(penalty=None, max_iter=1000).fit(X2, y)
b0, (b_glu, b_bmi) = model_2d.intercept_[0], model_2d.coef_[0]

print(f"log-odds = {b0:.3f} + {b_glu:.4f} x glucose + {b_bmi:.4f} x BMI")
print(f"decision boundary (log-odds = 0):  BMI = {-b0/b_bmi:.1f} - {b_glu/b_bmi:.3f} x glucose")
print(f"training accuracy at threshold 1/2: {(model_2d.predict(X2) == y).mean():.3f}")

In [ ]:
gg, bb = np.meshgrid(np.linspace(40, 210, 300), np.linspace(15, 70, 300))
pp = sigmoid(b0 + b_glu * gg + b_bmi * bb)

fig, ax = plt.subplots(figsize=(9, 6.5))
cs = ax.contour(gg, bb, pp, levels=[0.25, 0.5, 0.75], colors=["#2b6cb0", "#1a1a1a", "#c1272d"], linewidths=[1.5, 2.5, 1.5])
ax.clabel(cs, fmt={0.25: "P = 0.25", 0.5: "P = 0.5", 0.75: "P = 0.75"}, fontsize=10, manual=[(80, 48.2), (118, 44.4), (150, 43.2)])
for label, col, name in [(0, "#2b6cb0", "no diabetes"), (1, "#c1272d", "diabetes")]:
    m = y == label
    ax.scatter(X2[m, 0], X2[m, 1], s=14, alpha=0.5, color=col, label=name)
ax.set_xlabel("two-hour plasma glucose (mg/dL)"); ax.set_ylabel("BMI (kg/m²)")
ax.set_title("Decision boundary of a two-feature logistic regression")
ax.legend(loc="upper left"); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

Women who developed diabetes concentrate at high glucose and high BMI. The boundary slopes downward
because the two features trade off: a lower glucose is compensated by a higher BMI in the linear
score. The lines of equal probability are parallel to the boundary, as section 5 requires.

Many points lie on the wrong side of the boundary, and the training accuracy is 76%. The two classes
overlap in these two features — the same glucose and BMI are seen in women who did and did not
develop diabetes — and no line can separate them. A classifier's task is the best cut, not a perfect
one; what "best" means for a particular clinical use is Meeting 9.

## 6 — Fitting and reading the full model

The six-feature model is fitted and evaluated with the discipline of Meeting 7: a test set split off
before anything is fitted, and features standardised with training-set statistics.

> `stratify=y` makes `train_test_split` keep the same prevalence in the training and test sets.
> Without it, a random split of a small dataset can leave the two sets with different fractions of
> positive cases, and every prevalence-dependent quantity of Meeting 9 would then differ between
> them.

Standardising matters for reading the weights. On standardised features, $w_j$ is the change in
log-odds per standard deviation of feature $j$, and $e^{w_j}$ is the corresponding odds ratio, so the
weights can be compared across features.

### Measuring the fit

Two numbers are reported on each set:

- **accuracy**, the fraction of correct decisions at the threshold $1/2$
- the **cross-entropy per sample**, also called the log loss: the loss of section 4 divided by $n$,
  which scores the probabilities themselves rather than the decisions

Accuracy must be read against the prevalence. A classifier that predicts "no diabetes" for every
woman is correct for $65.5\%$ of this test set without using a single feature. Meeting 9 replaces
accuracy with quantities that keep the two kinds of error apart.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9400, stratify=y)
mu, sd = X_train.mean(axis=0), X_train.std(axis=0, ddof=1)
Z_train, Z_test = (X_train - mu) / sd, (X_test - mu) / sd

model = LogisticRegression(penalty=None, max_iter=1000).fit(Z_train, y_train)
p_train, p_test = model.predict_proba(Z_train)[:, 1], model.predict_proba(Z_test)[:, 1]

print(f"training: {len(y_train)} women (prevalence {100*y_train.mean():.1f}%)    "
      f"test: {len(y_test)} women (prevalence {100*y_test.mean():.1f}%)\n")
print(f"{'feature':<26}{'SD':>8}{'weight':>9}{'odds ratio per SD':>20}")
for name, s, w_j in zip(features, sd, model.coef_[0]):
    print(f"{name:<26}{s:>8.2f}{w_j:>9.3f}{np.exp(w_j):>20.2f}")
print(f"{'intercept':<26}{'':>8}{model.intercept_[0]:>9.3f}")

In [ ]:
def accuracy(y_true, p, threshold=0.5):
    return np.mean((p >= threshold) == y_true)

print(f"{'':<14}{'accuracy':>10}{'log loss':>10}")
print(f"{'training':<14}{accuracy(y_train, p_train):>10.3f}{log_loss(y_train, p_train):>10.3f}")
print(f"{'test':<14}{accuracy(y_test,  p_test):>10.3f}{log_loss(y_test,  p_test):>10.3f}")
print(f"\npredicting 'no diabetes' for everyone: test accuracy {1 - y_test.mean():.3f}")

acc = accuracy(y_test, p_test)
print(f"standard error of the test accuracy with {len(y_test)} women: about {np.sqrt(acc * (1 - acc) / len(y_test)):.3f}")

# scikit-learn's default adds an L2 penalty (C=1.0); on this data it barely moves the weights.
default = LogisticRegression(max_iter=1000).fit(Z_train, y_train)
print("\nweights, penalty=None :", np.round(model.coef_[0], 3))
print("weights, default C=1  :", np.round(default.coef_[0], 3))

Glucose carries the largest weight: one standard deviation of two-hour glucose, about 30 mg/dL,
multiplies the odds of diabetes by $2.8$ with the other five measurements held fixed. BMI (odds
ratio $1.6$ per $6.7$ kg/m²) and the number of pregnancies ($1.5$ per $3.4$) follow. Blood pressure
has an odds ratio of $0.93$: on its own it is correlated with the outcome, but given the other five
measurements it adds nothing, which is the same distinction as the weight of a collinear feature in
Meeting 7.

The test accuracy, $0.83$, is higher than the training accuracy, $0.77$. With 145 test women the
standard error of the test accuracy is about $0.03$, so the two are consistent with each other; a
single test set of this size cannot resolve differences of a few percentage points. Meeting 20
attaches confidence intervals to such numbers.

The last two lines show scikit-learn's default, which adds the ridge penalty of Meeting 7 to the
cross-entropy with a fixed strength (`C=1.0`, where a smaller `C` is a stronger penalty). On 579
training women and six features it changes the weights in the second decimal place. `penalty=None`
was used throughout this lecture so that the fitted weights are the maximum likelihood weights of
section 4; choosing the penalty strength by validation is Meeting 11.

## 7 — Connection to the course

**One recipe, twice.** Meeting 6 and today followed the same four steps: a model for how the labels
arise, the likelihood of the data under it, the negative log-likelihood as the loss, and the gradient
of the loss. Gaussian labels gave the squared loss and a closed-form solution; Bernoulli labels gave
the cross-entropy and an iterative one. Every model in the rest of the course is fitted by this
recipe, with gradient descent (Meeting 13) doing the minimisation.

**The logistic unit is the neuron.** A neural network layer (Meeting 17) computes many linear scores
at once and passes each through a nonlinearity; the sigmoid was the original choice, and a network
with a single unit *is* logistic regression. The cross-entropy is the loss of every classifier in the
course, including the convolutional networks of Meeting 21 and the language models of Meeting 25.

**More than two classes.** With $K$ classes the model keeps one weight vector $w_k$ per class and
replaces the sigmoid by the **softmax**,

$$P(y = k \mid x) = \frac{e^{x^{\top} w_k}}{\sum_{j=1}^{K} e^{x^{\top} w_j}}$$

which turns $K$ scores into $K$ probabilities that sum to one. The loss is unchanged in form —
$-\log$ of the probability assigned to the correct class — and the five cancer types of Meeting 5
are a problem of this kind.

**Regularisation carries over.** The ridge penalty of Meeting 7 is added to the cross-entropy loss
unchanged, and scikit-learn does so by default. Choosing its strength is the subject of Meeting 11.

## What to take away

- Logistic regression keeps the linear score $x^{\top} w$ of linear regression and passes it through
  the sigmoid: $P(y = 1 \mid x) = \sigma(x^{\top} w)$
- Equivalently, the log-odds of the positive class is linear in the features, and $e^{w_j}$ is the
  odds ratio per unit of feature $j$
- The cross-entropy loss is the negative log-likelihood of a Bernoulli model for the labels; its
  gradient is $X^{\top}(p - y)$, and it has no closed-form minimiser
- The decision boundary $x^{\top} w = 0$ is a hyperplane; the threshold $1/2$ that places it there is
  a choice
- Accuracy must be read against prevalence: a classifier that ignores the features is correct a
  fraction $1 - \text{prevalence}$ of the time

### Tuesday, October 13 — Meeting 9

The model fitted in section 6 is evaluated properly: the confusion matrix, sensitivity and
specificity, precision and recall, the ROC and precision–recall curves, the choice of threshold, and
what class imbalance does to each of them.

## Recommended resources

Everything here is optional and freely available. Each item covers material from this lecture in a
different style or voice; none of it is assigned.

### Start here

| Resource | Why this one | Time |
|---|---|---|
| **[StatQuest: Logistic Regression](https://www.youtube.com/watch?v=yIYKR4sgzI8)** — StatQuest | The model in nine minutes: the S-curve, the log-odds line, and what is being fitted. | 9 min |
| **[Odds and Log(Odds), Clearly Explained](https://www.youtube.com/watch?v=ARfXDSkQf1Y)** — StatQuest | The odds and the logit of section 3, slowly and with pictures. | 12 min |
| **[Odds Ratios and Log(Odds Ratios), Clearly Explained](https://www.youtube.com/watch?v=8nm0G-1uJzA)** — StatQuest | How a weight is read as an odds ratio, which is how the table of section 6 is read. | 16 min |
| **[Logistic Regression Details Pt1: Coefficients](https://www.youtube.com/watch?v=vN5cNN2-HWE)** — StatQuest | The weights of a fitted model and what "per unit" means for continuous and for categorical features. | 19 min |

### The mathematics

| Resource | Why this one | Time |
|---|---|---|
| **[CS229 Lecture Notes](https://cs229.stanford.edu/main_notes.pdf)** — Ma and Ng, Stanford, Chapter 2 | Section 2.1 derives logistic regression and its gradient as sections 3 and 4 do; section 2.3 treats the softmax of section 7. | 1 h |
| **[CS229 Lecture 3: Locally Weighted & Logistic Regression](https://www.youtube.com/watch?v=het9HFqo1TQ)** — Stanford, Autumn 2018 | The lecture that accompanies the notes; logistic regression and Newton's method occupy its second half. | 80 min |
| **[A Short Introduction to Entropy, Cross-Entropy and KL-Divergence](https://www.youtube.com/watch?v=ErfnhcEV1O8)** — Aurélien Géron | Where the name "cross-entropy" comes from, in eleven minutes. | 11 min |
| **[An Introduction to Statistical Learning](https://www.statlearning.com/)** — James, Witten, Hastie and Tibshirani, Chapter 4 | Sections 4.1 to 4.3: logistic regression from the statistician's side — estimating the weights, interpreting them, and regression on several features. Free PDF. | 2 h |

### Source of the data

| Resource | Why this one |
|---|---|
| **[Using the ADAP Learning Algorithm to Forecast the Onset of Diabetes Mellitus](https://pmc.ncbi.nlm.nih.gov/articles/PMC2245318/)** — Smith, Everhart, Dickson, Knowler and Johannes, 1988 | The paper that introduced the dataset: the 768 women, the five-year outcome, and a neural network of 1988. Free on PubMed Central. |